# Limpieza básica de los GRD respiratorios (2019-2025)

Este notebook toma los `GRD_PUBLICO_AAAA_ER.txt` (ya filtrados por `DIAGNOSTICO1` que empieza con `J`) y deja un `.parquet` limpio por año en `GRD_CIE_TM/GRD_limpio/`.

**Qué hace, en orden**

1. Lectura con la codificación correcta de cada año y limpieza de texto (espacios, vacíos).
2. Unificación de nombres de columnas entre años (`CIP_ENCRIPTADO` / `ID_BENEFICIARIO`).
3. **Duplicados** (con justificación).
4. **Nulos y valores "falsos nulos"** (con justificación).
5. Fechas a un único tipo `datetime64` (se muestran como `dd/mm/yyyy`).
6. Variables del proyecto: `SEXO`, `DIAGNOSTICO1`, `PROCEDIMIENTO1-30`, `FECHA_INGRESO`, `MES_INGRESO`, `TRAMO_EDAD`, `HOSPITAL`, `COMUNA`.
7. Reporte de calidad (antes/después) y exportación a parquet.

> **Ojo con el año:** cada archivo `GRD_PUBLICO_AAAA` contiene los egresos con **fecha de alta** en ese año; por eso ~1% de los `FECHA_INGRESO` caen en el año anterior (el 2025 llega hasta 2023). Para análisis temporal por mes de ingreso, hay que concatenar todos los años y agrupar por `FECHA_INGRESO`, no por `ANIO`.

> Las variables auxiliares `EDAD` y `FECHA_NACIMIENTO` se usan solo para calcular `TRAMO_EDAD` y se eliminan antes de formar el GRD final.

In [20]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------- Configuración ----------------
DIR_GRD     = Path('../GRD_CIE_TM')

ANIOS = range(2019, 2026)

# Codificación real de cada archivo _ER.txt (verificada: 2019-2024 NO son utf-8 válido; 2025 sí lo es).
ENCODING = {anio: 'latin-1' for anio in ANIOS}
ENCODING[2025] = 'utf-8'

# Si True, FECHA_INGRESO se guarda como texto 'dd/mm/yyyy'.
# Recomendado False: en parquet se guardan como fecha real (se pueden filtrar y ordenar) y
# el formato dd/mm/yyyy se aplica solo al mostrarlas (ver última celda).
FECHAS_COMO_TEXTO = False

## 1. Catálogos (CIE-10, CIE-9, tabla maestra de hospitales)

Dos detalles encontrados al revisar los datos:

* **CIE-9**: el catálogo trae el código con dos dígitos enteros (`00.17`, `03.31`), pero el GRD trae `0.17`, `3.31`. Sin corregir esto, esos procedimientos no calzarían. Se normaliza con un cero a la izquierda.
* **Hospitales**: la tabla maestra solo trae *código* y *nombre*, no la comuna.

In [21]:
def limpiar_texto(s):
    """strip + vacíos -> NA"""
    s = s.astype('string').str.strip()
    return s.mask(s == '')

def normalizar_cie9(s):
    """'0.17' -> '00.17' ; '3.9' -> '03.9' ; '28.0' queda igual."""
    return limpiar_texto(s).str.replace(r'^(\d)(?=\.|$)', r'0\1', regex=True)

# --- CIE-10 (diagnósticos) ---
cie10 = pd.read_excel(DIR_GRD / 'CIE-10.xlsx', dtype=str)
cie10['Código'] = limpiar_texto(cie10['Código']).str.upper()
DESC_CIE10 = cie10.drop_duplicates('Código').set_index('Código')['Descripción']

# --- CIE-9 (procedimientos) ---
cie9 = pd.read_excel(DIR_GRD / 'CIE-9 .xlsx', dtype=str)
cie9['Código'] = normalizar_cie9(cie9['Código'])
DESC_CIE9 = (cie9.drop_duplicates('Código')
                 .set_index('Código')['Descripción']
                 .str.strip().str.capitalize())          # viene TODO EN MAYÚSCULAS

# --- Tabla maestra de hospitales ---
maestra = pd.read_excel(DIR_GRD / 'Tablas maestras bases GRD .xlsx', sheet_name='Hospitales')
maestra.columns = ['COD_HOSPITAL', 'HOSPITAL']
maestra['COD_HOSPITAL'] = limpiar_texto(maestra['COD_HOSPITAL'])
maestra['HOSPITAL'] = limpiar_texto(maestra['HOSPITAL'])
NOMBRE_HOSPITAL = maestra.drop_duplicates('COD_HOSPITAL').set_index('COD_HOSPITAL')['HOSPITAL']



## 2. Comuna del hospital

**Problema:** la columna `COMUNA` de los GRD 2019-2024 es la comuna de **residencia del paciente**, no la del hospital (por ejemplo, un paciente de Arauco atendido en el hospital de Concepción). Por eso, "deducir" la comuna del hospital con la comuna más frecuente de sus pacientes **falla** en ~1 de cada 3 hospitales (Barros Luco quedaría en "Lo Espejo" y no en San Miguel; el Hospital de Niños Roberto del Río en "Quilicura" y no en Independencia).

**Solución usada:** la tabla maestra trae la comuna entre paréntesis en 53 de los 76 hospitales presentes en los datos (`"Hospital San José (Melipilla)"`, `"... (Santiago, San Miguel)"` → la última palabra es la comuna). Los 22 restantes se completan a mano en `COMUNA_MANUAL`.

> ⚠️ `COMUNA_MANUAL` está hecho de memoria y **debe ser validado** con el catálogo oficial de establecimientos del DEIS (MINSAL). El código `200717` (aparece solo en 2024-2025, ~1.700 filas, Servicio de Salud Araucanía Sur) **no está en la maestra**: queda sin comuna hasta identificarlo.

In [22]:
# Hospitales cuyo nombre en la tabla maestra NO trae comuna entre paréntesis. VALIDAR con DEIS.
COMUNA_MANUAL = {
    '107102': 'QUILPUÉ',      '110110': 'QUINTA NORMAL', '111101': 'MAIPÚ',
    '111195': 'SANTIAGO',     '112103': 'PROVIDENCIA',   '112104': 'PROVIDENCIA',
    '114105': 'LA FLORIDA',   '115100': 'RANCAGUA',      '115110': 'SANTA CRUZ',
    '116107': 'CONSTITUCIÓN', '116109': 'SAN JAVIER',    '117102': 'SAN CARLOS',
    '118106': 'LOTA',         '119101': 'TOMÉ',          '119102': 'PENCO',
    '121114': 'NUEVA IMPERIAL','121117': 'PITRUFQUÉN',   '121121': 'VILLARRICA',
    '123100': 'OSORNO',       '124105': 'PUERTO MONTT',  '126100': 'PUNTA ARENAS',
    '133150': 'CASTRO', '200717' : 'PADRE LAS CASAS',
}

def comuna_desde_nombre(nombre):
    """'Hospital X (Santiago, San Miguel)' -> 'SAN MIGUEL' ; 'Hospital Y (Melipilla)' -> 'MELIPILLA'"""
    if pd.isna(nombre):
        return np.nan
    m = re.search(r'\(([^)]*)\)\s*$', nombre)
    return m.group(1).split(',')[-1].strip().upper() if m else np.nan

COMUNA_HOSPITAL = NOMBRE_HOSPITAL.map(comuna_desde_nombre)
for cod, comuna in COMUNA_MANUAL.items():
    COMUNA_HOSPITAL.loc[cod] = comuna


## 3. Funciones de limpieza

### Decisiones y justificación

**Duplicados**
* Se **eliminan solo los duplicados exactos** (las 129 columnas idénticas). Un registro idéntico en todo, incluyendo fechas de traslado, alta, GRD y médico encriptado, es un error de extracción, no dos hospitalizaciones reales. Esto afecta casi solo a 2020 (~128 filas) y 2022 (2 filas).
* Los registros que coinciden en `ID + hospital + fecha de ingreso + DIAGNOSTICO1` pero difieren en otras columnas se consideran posibles duplicados y **se eliminan**. El total eliminado queda registrado en el reporte de calidad.

**Nulos**
* **Falsos nulos de texto**: `DESCONOCIDO`/`DESCONOCIDA`/`NO APLICA` dentro de columnas de fecha o de procedimiento se convierten en nulo real.
* **Nulos estructurales** (`PROCEDIMIENTO2-30`, `DIAGNOSTICO2-35`, `FECHATRASLADO*`, `*RN*`): el vacío significa "no aplica" (el paciente no tuvo ese procedimiento o traslado). **No se imputan ni se eliminan filas.** Incluye `PROCEDIMIENTO1` (casi el 100% de sus nulos son pacientes sin procedimientos).
* **Falsos nulos** (`DESCONOCIDO`, `NO APLICA` dentro de columnas de fecha): se convierten a nulo real (`NaT`), porque una fecha no puede ser "desconocido".
* **`SEXO = DESCONOCIDO`** (<0.02% de las filas): se conserva como categoría explícita. Es información, y es consistente con `TRAMO_EDAD = DESCONOCIDO` del GRD 2025.
* Las filas con `FECHA_INGRESO` nula se eliminan, porque `MES_INGRESO` depende de una fecha válida. Los nulos de identificación y fechas de nacimiento se informan sin imputarlos.

In [23]:
FALSOS_NULOS = {'DESCONOCIDO', 'DESCONOCIDA', 'NO APLICA', 'NO INFORMADO', 'S/I', 'SIN INFORMACION', 'SIN INFORMACIÓN'}

COLS_NUM_COMA = ['IR_29301_PESO', 'PESORN1', 'PESORN2', 'PESORN3', 'PESORN4']
COLS_NUM = ['RN1ESTADO', 'RN2ESTADO', 'RN3ESTADO', 'RN4ESTADO', 'USOSPABELLON',
            'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD']

ETIQUETAS_TRAMO = [f'[{a} - {a + 4} años]' for a in range(0, 110, 5)]   # [0 - 4 años] ... [105 - 109 años]
BINS_TRAMO = list(range(0, 111, 5))


def parsear_fecha(serie):
    """Acepta 'yyyy-mm-dd' y 'dd-mm-yyyy' (formato por año/columna distinto). Lo demás -> NaT."""
    s = limpiar_texto(serie)
    s = s.mask(s.str.upper().isin(FALSOS_NULOS))
    es_iso = s.str.match(r'^\d{4}-\d{2}-\d{2}').fillna(False).astype(bool)
    es_dmy = s.str.match(r'^\d{2}-\d{2}-\d{4}').fillna(False).astype(bool)
    f_iso = pd.to_datetime(s.where(es_iso).str[:10], format='%Y-%m-%d', errors='coerce')
    f_dmy = pd.to_datetime(s.where(es_dmy).str[:10], format='%d-%m-%Y', errors='coerce')
    return f_iso.fillna(f_dmy)


def calcular_edad(f_nac, f_ing):
    """Edad en años cumplidos al ingreso. Fuera de 0-109 -> NaN."""
    edad = f_ing.dt.year - f_nac.dt.year
    aun_no_cumple = ((f_ing.dt.month < f_nac.dt.month) |
                     ((f_ing.dt.month == f_nac.dt.month) & (f_ing.dt.day < f_nac.dt.day)))
    edad = edad - aun_no_cumple.astype(int)
    return edad.where((edad >= 0) & (edad <= 109))


def a_tramo(edad):
    """Mismos intervalos del GRD 2025."""
    tramo = pd.cut(edad, bins=BINS_TRAMO, right=False, labels=ETIQUETAS_TRAMO)
    return tramo.astype('string').fillna('DESCONOCIDO')


def limpiar_grd(anio):
    ruta = DIR_GRD / f'GRD_PUBLICO_{anio}_ER.txt'
    df = pd.read_csv(ruta, sep='|', encoding=ENCODING[anio], dtype=str)
    rep = {'anio': anio, 'filas_originales': len(df)}

    # --- 1. Texto: strip y vacíos -> NA ---
    for c in df.columns:
        df[c] = limpiar_texto(df[c])

    # --- 2. Unificar nombre del ID (2019-2023: CIP_ENCRIPTADO; 2024-2025: ID_BENEFICIARIO) ---
    df = df.rename(columns={'CIP_ENCRIPTADO': 'ID_BENEFICIARIO'})

    # --- 3. Duplicados exactos ---
    n = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    rep['duplicados_exactos_eliminados'] = n - len(df)

    # --- 4. Fechas (todas las columnas FECHA*) -> datetime64 ---
    cols_fecha = [c for c in df.columns if c.startswith('FECHA')]
    for c in cols_fecha:
        crudo = df[c]
        df[c] = parsear_fecha(crudo).astype('datetime64[ns]')     # misma unidad en todos los años
        malas = crudo.notna() & df[c].isna() & ~crudo.str.upper().isin(FALSOS_NULOS)
        if malas.any():
            rep[f'fecha_no_parseable_{c}'] = int(malas.sum())
    rep['FECHA_INGRESO_nula'] = int(df['FECHA_INGRESO'].isna().sum())
    df = df.dropna(subset=['FECHA_INGRESO']).reset_index(drop=True)
    df['MES_INGRESO'] = df['FECHA_INGRESO'].dt.month.astype('Int64')
    rep['FECHA_INGRESO_fuera_del_anio'] = int((df['FECHA_INGRESO'].dt.year != anio).sum() - df['FECHA_INGRESO'].isna().sum())

    # --- 5. Numéricos (coma decimal, ceros a la izquierda del código GRD) ---
    for c in COLS_NUM_COMA:
        if c in df: df[c] = pd.to_numeric(df[c].str.replace(',', '.', regex=False), errors='coerce')
    for c in COLS_NUM:
        if c in df: df[c] = pd.to_numeric(df[c], errors='coerce')
    df['IR_29301_COD_GRD'] = df['IR_29301_COD_GRD'].str.lstrip('0')   # '044163' (2019-23) vs '41203' (2025)

    # --- 6. ANIO ---
    df.insert(0, 'ANIO', anio)

    # --- 7. SEXO ---
    df['SEXO'] = df['SEXO'].str.upper()
    df.loc[~df['SEXO'].isin(['HOMBRE', 'MUJER']), 'SEXO'] = 'DESCONOCIDO'

    # --- 8. Diagnóstico y procedimientos: código -> palabras ---
    df['DIAGNOSTICO1'] = df['DIAGNOSTICO1'].str.upper()
    df['DIAGNOSTICO1_DESC'] = df['DIAGNOSTICO1'].map(DESC_CIE10)
    rep['DIAGNOSTICO1_sin_descripcion'] = int(df['DIAGNOSTICO1_DESC'].isna().sum())

    cols_proc = [f'PROCEDIMIENTO{i}' for i in range(1, 31)]
    sin_match = pd.Series(dtype='string')
    for c in cols_proc:
        df[c] = normalizar_cie9(df[c])
        df[c] = df[c].mask(df[c].str.upper().isin(FALSOS_NULOS))      # 'DESCONOCIDO' en PROCEDIMIENTO* = sin dato
        df[f'{c}_DESC'] = df[c].map(DESC_CIE9)
        sin_match = pd.concat([sin_match, df.loc[df[c].notna() & df[f'{c}_DESC'].isna(), c]])
    rep['procedimientos_con_codigo'] = int(df[cols_proc].notna().sum().sum())
    rep['procedimientos_sin_descripcion'] = len(sin_match)
    rep['procedimientos_sin_desc_top3'] = ', '.join(sin_match.value_counts().head(3).index)

    # --- 9. Posible duplicado (se marca y se elimina) ---
    clave = ['ID_BENEFICIARIO', 'COD_HOSPITAL', 'FECHA_INGRESO', 'DIAGNOSTICO1']
    df['POSIBLE_DUPLICADO'] = df.duplicated(clave, keep=False) & df['ID_BENEFICIARIO'].notna()
    rep['posibles_duplicados_marcados'] = int(df['POSIBLE_DUPLICADO'].sum())
    df = df.loc[~df['POSIBLE_DUPLICADO']].reset_index(drop=True)
    rep['posibles_duplicados_eliminados'] = rep['posibles_duplicados_marcados']

    # --- 10. Edad / TRAMO_EDAD ---
    if 'FECHA_NACIMIENTO' not in df:                      # GRD 2025: no viene la fecha de nacimiento
        df['FECHA_NACIMIENTO'] = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
    if 'TRAMO_EDAD' in df:                                # 2025: se respeta el tramo original
        df['EDAD'] = pd.array([pd.NA] * len(df), dtype='Int64')
        df['TRAMO_EDAD'] = df['TRAMO_EDAD'].str.strip()
        invalidos = ~df['TRAMO_EDAD'].isin(ETIQUETAS_TRAMO + ['DESCONOCIDO'])
        rep['TRAMO_EDAD_etiquetas_inesperadas'] = int(invalidos.sum())
    else:                                                 # 2019-2024: se calcula
        edad = calcular_edad(df['FECHA_NACIMIENTO'], df['FECHA_INGRESO'])
        df['EDAD'] = edad.astype('Int64')
        df['TRAMO_EDAD'] = a_tramo(edad)
    rep['TRAMO_EDAD_desconocido'] = int((df['TRAMO_EDAD'] == 'DESCONOCIDO').sum())
    df = df.drop(columns=['EDAD', 'FECHA_NACIMIENTO'])

    # --- 11. Hospital y comuna ---
    df['COD_HOSPITAL'] = df['COD_HOSPITAL'].str.strip()
    df['HOSPITAL'] = df['COD_HOSPITAL'].map(NOMBRE_HOSPITAL)
    rep['hospital_sin_maestra'] = int(df['HOSPITAL'].isna().sum())
    rep['hospital_sin_maestra_cods'] = ', '.join(df.loc[df['HOSPITAL'].isna(), 'COD_HOSPITAL'].dropna().unique())
    df['HOSPITAL'] = df['HOSPITAL'].fillna('SIN CATALOGAR (' + df['COD_HOSPITAL'] + ')')

    # COMUNA = comuna DEL HOSPITAL (única definición posible en los 7 años, porque 2025 no trae comuna de residencia).
    # La comuna de residencia original (2019-2024) se conserva aparte.
    if 'COMUNA' in df:
        df = df.rename(columns={'COMUNA': 'COMUNA_RESIDENCIA'})
    else:
        df['COMUNA_RESIDENCIA'] = pd.NA
    df['COMUNA'] = df['COD_HOSPITAL'].map(COMUNA_HOSPITAL)
    rep['COMUNA_hospital_nula'] = int(df['COMUNA'].isna().sum())

    # --- 12. Orden: variables clave primero ---
    clave_primero = ['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'COMUNA_RESIDENCIA', 'ID_BENEFICIARIO', 'SEXO',
                     'TRAMO_EDAD', 'FECHA_INGRESO', 'MES_INGRESO', 'FECHAALTA',
                     'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC', 'POSIBLE_DUPLICADO']
    resto = [c for c in df.columns if c not in clave_primero]
    df = df[clave_primero + resto]

    rep['filas_finales'] = len(df)
    return df, rep

## 4. Ejecutar todos los años y exportar

Se procesa un año a la vez (y se libera memoria) porque los archivos son grandes. Con `FECHAS_COMO_TEXTO = True` las fechas se guardan como texto `dd/mm/yyyy`.

In [24]:
reportes = []
grd = {}   # solo las columnas de interés, en memoria (nada se escribe a disco)

COLS_VER = (['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'SEXO', 'TRAMO_EDAD', 'FECHA_INGRESO', 
             'MES_INGRESO', 'FECHAALTA', 'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC']
            + [f'PROCEDIMIENTO{i}' for i in range(1, 31)]
            + [f'PROCEDIMIENTO{i}_DESC' for i in range(1, 31)])

for anio in ANIOS:
    df, rep = limpiar_grd(anio)

    if FECHAS_COMO_TEXTO:
        df['FECHA_INGRESO'] = df['FECHA_INGRESO'].dt.strftime('%d/%m/%Y')

    reportes.append(rep)
    grd[anio] = df[COLS_VER].copy()
    print(f'{anio}: {rep["filas_originales"]:>7,} -> {rep["filas_finales"]:>7,} filas | '
          f'duplicados exactos eliminados: {rep["duplicados_exactos_eliminados"]}')
    del df

reporte = pd.DataFrame(reportes).set_index('anio').T
reporte

# Cantidad de valores nulos por columna en el GRD final de cada año.
nulos_por_anio = pd.DataFrame({anio: datos.isna().sum() for anio, datos in grd.items()})
nulos_por_anio

2019:  96,439 ->  96,379 filas | duplicados exactos eliminados: 0
2020:  34,962 ->  34,800 filas | duplicados exactos eliminados: 128
2021:  34,618 ->  34,612 filas | duplicados exactos eliminados: 0
2022:  66,636 ->  66,622 filas | duplicados exactos eliminados: 2
2023:  88,782 ->  88,766 filas | duplicados exactos eliminados: 0
2024:  88,764 ->  88,757 filas | duplicados exactos eliminados: 0
2025:  94,161 ->  94,147 filas | duplicados exactos eliminados: 0


,2019,2020,2021,2022,2023,2024,2025
ANIO,0,0,0,0,0,0,0
COD_HOSPITAL,0,0,0,0,0,0,0
HOSPITAL,0,0,0,0,0,0,0
COMUNA,0,0,0,0,0,0,0
SEXO,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...
PROCEDIMIENTO26_DESC,94241,33158,32756,63710,84930,84638,89888
PROCEDIMIENTO27_DESC,94548,33382,33023,64122,85474,85165,90482
PROCEDIMIENTO28_DESC,94850,33581,33217,64459,85949,85705,90958
PROCEDIMIENTO29_DESC,95089,33753,33417,64759,86345,86104,91384


## 5. Distribuciones de variables categóricas

Se muestran todos los valores distintos y su cantidad por año para `SEXO`, `COMUNA` y `MES_INGRESO`.

In [25]:
for columna in ['SEXO', 'COMUNA', 'MES_INGRESO']:
    print(f'\n{columna}')
    print(
        pd.concat(
            [datos[columna].value_counts(dropna=False).rename(anio) for anio, datos in grd.items()],
            axis=1,
        ).fillna(0).astype('int64').sort_index()
    )



SEXO
              2019   2020   2021   2022   2023   2024   2025
SEXO                                                        
DESCONOCIDO     18      4      1      9     10      6      6
HOMBRE       50180  18979  18999  35481  46730  46041  48943
MUJER        46181  15817  15612  31132  42026  42710  45198

COMUNA
              2019  2020  2021  2022  2023  2024  2025
COMUNA                                                
ANGOL          923   399   376   911  1238  1002  1115
ANTOFAGASTA   1493   927   969  1084  1136  1330  1279
ARICA          975   521   506   708   788   745   774
BUIN           858   229   263   496   600   572   588
CALAMA        1041   564   484   913  1252  1089  1169
...            ...   ...   ...   ...   ...   ...   ...
VALLENAR       546   269   238   352   471   487   521
VALPARAÍSO    3112   850   855  1964  2604  2549  2356
VICTORIA       803   427   405   729   858   896  1007
VILLARRICA     458   180   230   448   479   482   656
VIÑA DEL MAR  1633   

## 6. Verificación rápida

Se comprueban invariantes del GRD final: fecha de ingreso válida, mes consistente, ausencia de variables auxiliares y tramos válidos.

In [26]:
chk = pd.concat(grd.values(), ignore_index=True)
fecha_valida = chk['FECHA_INGRESO'].notna().all()
mes_consistente = chk['MES_INGRESO'].equals(chk['FECHA_INGRESO'].dt.month.astype('Int64'))
sin_auxiliares = not {'EDAD', 'FECHA_NACIMIENTO'}.intersection(chk.columns)
tramos_validos = set(chk['TRAMO_EDAD'].dropna().unique()) <= set(ETIQUETAS_TRAMO + ['DESCONOCIDO'])

print(f'Fecha de ingreso sin nulos: {fecha_valida}')
print(f'MES_INGRESO coincide con FECHA_INGRESO: {mes_consistente}')
print(f'EDAD y FECHA_NACIMIENTO no están en el GRD final: {sin_auxiliares}')
print(f'TRAMO_EDAD contiene solo valores válidos: {tramos_validos}')

assert fecha_valida
assert mes_consistente
assert sin_auxiliares
assert tramos_validos
print(f'\nValidacion OK: {len(chk):,} filas y {chk.shape[1]} columnas')


Fecha de ingreso sin nulos: True
MES_INGRESO coincide con FECHA_INGRESO: True
EDAD y FECHA_NACIMIENTO no están en el GRD final: True
TRAMO_EDAD contiene solo valores válidos: True

Validacion OK: 504,083 filas y 71 columnas
